# Multi-Agent Deep Q-Network (DQN) Coordination in a Grid World

## Project Overview

This project implements a **Multi-Agent Deep Q-Network (DQN)** to solve a cooperative transportation task in a **5×5 Grid World**. Four autonomous agents repeatedly transport items between a pickup location (**A**) and a drop-off location (**B**) while learning to coordinate their movements and avoid head-on collisions.

Unlike traditional single-agent reinforcement learning, this project focuses on **multi-agent coordination**, where each agent must make decisions that not only maximize its own reward but also contribute to efficient group behavior.

The project is implemented **from scratch** without using reinforcement learning libraries. PyTorch (or TensorFlow) is used only to build the Deep Q-Network.

---

# Objectives

The primary objectives of this project are:

- Build a custom 5×5 Grid World environment.
- Simulate four independent agents.
- Train agents using the Deep Q-Network (DQN) algorithm.
- Learn efficient transportation between pickup and drop-off locations.
- Minimize head-on collisions through learned coordination.
- Evaluate the trained agents on unseen scenarios.

---

# Problem Description

Each agent continuously performs the following cycle:

1. Move toward the pickup location (**A**).
2. Pick up an item automatically.
3. Move toward the drop-off location (**B**).
4. Deliver the item automatically.
5. Return to the pickup location.
6. Repeat the process.

Agents move simultaneously (executed sequentially in random order each timestep) and must learn to coordinate their actions to avoid collisions while maintaining efficient deliveries.

---

# Environment Specifications

| Property | Value |
|-----------|-------|
| Grid Size | 5 × 5 |
| Number of Agents | 4 |
| Pickup Location | A |
| Drop-off Location | B |
| Action Space | North, South, East, West |
| Wait Action | Not Allowed |
| Movement Order | Random Sequential |
| Learning Algorithm | Deep Q-Network (DQN) |

---

# Observation Space

Each agent observes:

- Its current position
- Pickup location (A)
- Drop-off location (B)
- Whether it is currently carrying an item

The observation is converted into a numerical state vector before being passed to the neural network.

---

# Action Space

Each agent can perform one of four actions:

| Action | Description |
|----------|-------------|
| North | Move one cell upward |
| South | Move one cell downward |
| East | Move one cell to the right |
| West | Move one cell to the left |

No **Wait** action is allowed.

---

# Head-On Collision Rule

A collision occurs **only** when:

- One agent is travelling **from A to B**, and
- Another agent is travelling **from B to A**, and
- Both occupy the same cell after movement.

Special cases:

- Multiple agents may occupy the same cell if they are travelling in the same direction.
- Collisions occurring on the pickup location (A) or drop-off location (B) are ignored.

---

# Reinforcement Learning Approach

This project uses the **Deep Q-Network (DQN)** algorithm.

Unlike Tabular Q-Learning, which stores Q-values inside a lookup table, DQN approximates the action-value function using a neural network.

The network predicts the expected future reward for every possible action given the current state.

---

# DQN Learning Equation

The target Q-value is computed using

\[
Q(s_t,a_t)=r_t+\gamma\max_aQ(s_{t+1},a)
\]

The neural network is trained to minimize the difference between the predicted Q-value and the target Q-value.

---

# Project Components

The implementation is divided into several stages:

1. Environment Construction
2. Agent Representation
3. Collision Detection
4. State Representation
5. Replay Memory
6. Deep Q-Network
7. DQN Agent
8. Training Loop
9. Performance Evaluation
10. Visualization

---

# Training Constraints

The implementation follows the assignment requirements.

| Constraint | Limit |
|------------|-------|
| Maximum Training Steps | 1,500,000 |
| Maximum Collisions | 4,000 |
| Maximum Training Time | 10 minutes |

---

# Performance Goal

After training, the system should:

- Complete a full delivery cycle successfully.
- Finish within **25 steps**.
- Remain collision-free.
- Achieve at least **75% success rate** over multiple evaluation scenarios.

---

# Notebook Structure

This notebook is organized as follows:

1. Import Libraries
2. Define Constants and Hyperparameters
3. Define Enumerations
4. Create Agent Class
5. Build Grid World Environment
6. Implement Collision Detection
7. Define Reward Function
8. Build Replay Buffer
9. Create Deep Q-Network
10. Implement DQN Agent
11. Train the Agents
12. Evaluate Performance
13. Visualize Training Results
14. Demonstrate

# 1. Import Libraries

In [1]:
import numpy as np
from dataclasses import dataclass
from enum import Enum

# 2. Environment Constants

In [2]:
GRID_SIZE = 5
AGENT_COUNT = 4

PICKUP = (0, 0)
DROPOFF = (4, 4)

# 3. Action Enumeration

In [3]:
class Direction(Enum):
    NORTH = (-1, 0)
    SOUTH = (1, 0)
    WEST = (0, -1)
    EAST = (0, 1)

# 4. Agent Representation

In [5]:
@dataclass
class Agent:
    id: int
    position: tuple[int, int]
    carrying: bool = False

# 5. Grid World Environment

Multi-agent 5x5 Grid World environment.

    Responsibilities:
    - Initialize the environment
    - Spawn agents
    - Move agents
    - Handle pickup and delivery
    - Return the current state of all agents

In [6]:
class GridWorld:
    # ------------------------------------------------------------------
    # Constructor
    # ------------------------------------------------------------------
    def __init__(self):
        self.size = GRID_SIZE
        self.pickup = PICKUP
        self.dropoff = DROPOFF
        self.reset()

    # ------------------------------------------------------------------
    # Generate a random position inside the grid
    # ------------------------------------------------------------------
    def random_position(self):
        row = np.random.randint(self.size)
        col = np.random.randint(self.size)
        return (row, col)

    # ------------------------------------------------------------------
    # Create all agents with random starting positions
    # ------------------------------------------------------------------
    def create_agents(self):
        self.agents = []

        for agent_id in range(AGENT_COUNT):
            position = self.random_position()

            agent = Agent(
                id=agent_id,
                position=position,
                carrying=False
            )

            self.agents.append(agent)

    # ------------------------------------------------------------------
    # Reset the environment to the initial state
    # ------------------------------------------------------------------
    def reset(self):
        self.steps = 0
        self.collisions = 0

        self.create_agents()

        return self.get_states()

    # ------------------------------------------------------------------
    # Move a position according to an action
    # Agents cannot leave the grid
    # ------------------------------------------------------------------
    def move(self, position, action):
        dr, dc = action.value

        row = position[0] + dr
        col = position[1] + dc

        row = np.clip(row, 0, self.size - 1)
        col = np.clip(col, 0, self.size - 1)

        return (row, col)

    # ------------------------------------------------------------------
    # Automatically pick up an item when reaching location A
    # ------------------------------------------------------------------
    def pickup_item(self, agent):
        if agent.carrying:
            return

        if agent.position != self.pickup:
            return

        agent.carrying = True

    # ------------------------------------------------------------------
    # Automatically deliver an item when reaching location B
    # ------------------------------------------------------------------
    def deliver_item(self, agent):
        if not agent.carrying:
            return

        if agent.position != self.dropoff:
            return

        agent.carrying = False

    # ------------------------------------------------------------------
    # Update a single agent after choosing an action
    # ------------------------------------------------------------------
    def update_agent(self, agent, action):
        agent.position = self.move(agent.position, action)

        self.pickup_item(agent)
        self.deliver_item(agent)

    # ------------------------------------------------------------------
    # Execute one environment step for one agent
    # ------------------------------------------------------------------
    def execute_action(self, agent_index, action):

        agent = self.agents[agent_index]
    
        old_carrying = agent.carrying
    
        # Move the agent
        agent.position = self.move(agent.position, action)
    
        # Pickup / Dropoff
        self.pickup(agent)
        self.dropoff(agent)
    
        picked_up = (not old_carrying) and agent.carrying
        delivered = old_carrying and (not agent.carrying)
    
        # Collision detection
        collision = self.is_collision(agent)
    
        if collision:
            self.collisions += 1
    
        reward = self.calculate_reward(
            collision,
            picked_up,
            delivered
        )
    
        self.steps += 1
    
        next_state = (
            agent.position,
            self.pickup,
            self.dropoff,
            agent.carrying
        )
    
        done = False
    
        return next_state, reward, done

    # ------------------------------------------------------------------
    # Return the observation for every agent
    # ------------------------------------------------------------------
    def get_states(self):
        states = []

        for agent in self.agents:
            state = (
                agent.position,
                self.pickup,
                self.dropoff,
                agent.carrying
            )

            states.append(state)

        return states